# Train (Kaggle T4×2)

LoRA fine-tune a small open-weight LLM on your attack trajectories using
Unsloth, then export to GGUF for Ollama. Runs on **Kaggle Notebooks** with
the **GPU T4 x2** accelerator — 32 GB total VRAM, enough for Llama-3.2-3B
(fits a single T4) or Llama-3.1-8B (uses both via `device_map="auto"`).

**Before you start:**
1. Notebook → Settings → Accelerator → **GPU T4 x2**
2. Internet → **On**
3. Upload your `alpaca.jsonl` as a Kaggle dataset, or clone the studio repo as
   shown below.

The two T4s have 15 GB VRAM each. For the 3B default, one T4 is plenty and
the second GPU stays idle — that's fine. For 8B+ models, `device_map="auto"`
shards automatically.

In [ ]:
!nvidia-smi

In [ ]:
# Kaggle ships CUDA 12.x but ships torch 2.x compatible with it.
# Unsloth pins versions tightly — install clean.
!pip install -q -U unsloth datasets accelerate peft bitsandbytes transformers trl

In [ ]:
import os
WORK_DIR = '/kaggle/working/jailbreak-pipeline-studio'
if not os.path.isdir(WORK_DIR):
    !git clone https://github.com/rkstlohchab/jailbreak-pipeline-studio.git $WORK_DIR
else:
    !cd $WORK_DIR && git pull --ff-only
os.chdir(WORK_DIR)
!pwd

In [ ]:
# Dataset location:
#   Option A — use the committed alpaca.jsonl (if the studio has one at studio/data/exports/)
#   Option B — upload alpaca.jsonl as a Kaggle dataset and set DATASET below to that mount path
#              e.g. /kaggle/input/my-alpaca/alpaca.jsonl
DATASET = 'studio/scripts/_sample_alpaca.jsonl'  # smoke-test with sample; swap for real export

import json
with open(DATASET) as f:
    rows = [json.loads(l) for l in f if l.strip()]
print(f'rows: {len(rows)}')
print(f'first: {rows[0]}')

In [ ]:
# Train. Writes LoRA adapter to studio/data/finetunes/<name>/.
# On T4×2 with 3B base + 500 examples + 2 epochs → ~20-30 min.
!python studio/scripts/train_lora.py \
  --model unsloth/Llama-3.2-3B-Instruct-bnb-4bit \
  --dataset $DATASET \
  --output /kaggle/working/finetune-llama3.2-3b \
  --epochs 2 \
  --lora-r 16 --lora-alpha 32 \
  --batch-size 2 --grad-accum 4 \
  --max-seq-length 2048

In [ ]:
# Merge LoRA + convert to GGUF for Ollama.
# llama.cpp needs to be cloned + built for the convert_hf_to_gguf.py script.
!git clone --depth 1 https://github.com/ggerganov/llama.cpp.git /kaggle/working/llama.cpp
!pip install -q -r /kaggle/working/llama.cpp/requirements.txt
!cd /kaggle/working/llama.cpp && cmake -B build && cmake --build build --config Release -j 4 2>&1 | tail -3
import os
os.environ['LLAMA_CPP_DIR'] = '/kaggle/working/llama.cpp'

!python studio/scripts/export_gguf.py \
  --adapter /kaggle/working/finetune-llama3.2-3b \
  --base unsloth/Llama-3.2-3B-Instruct \
  --output /kaggle/working/finetune-llama3.2-3b.gguf \
  --quant q4_k_m

In [ ]:
# The GGUF + a Modelfile template are in /kaggle/working/.
# Click the Output panel on the right to download. Then locally:
#   ollama create jb-llama3.2 -f Modelfile
#   ollama run jb-llama3.2 "your prompt"
# ...and point /attack at it: provider=__custom__, base URL=http://localhost:11434/v1, model=jb-llama3.2
!ls -lh /kaggle/working/*.gguf 2>/dev/null
!ls -lh /kaggle/working/finetune-llama3.2-3b/

## Next steps

1. Download `finetune-llama3.2-3b.gguf` from Kaggle's Output panel.
2. On your Mac, create an Ollama model from it:
   ```bash
   ollama create jb-llama3.2 -f Modelfile
   ```
3. Test it in the studio:
   - Open `/attack`
   - Target config: provider = `__custom__`, base URL = `http://localhost:11434/v1`, model = `jb-llama3.2`, api_key = `ollama`
4. Benchmark the before/after shift in `/eval`: run the default battery against the base model, then against your fine-tune. Compare view shows per-technique deltas.